# 作业 02：把多个输入接入上周的训练模板

作业 01 比较了同一个样本的两种表示：两个标量输入张量，或一个包含两个数的输入张量。本题把多个样本分别放入 x1、x2，练习补全训练代码。

请参考上周模板，让同一个预测函数同时使用 x1 和 x2。所有样本共享 w1、w2、b；每次预测都需要两个输入。下面提供函数和数据，参数创建、损失、梯度与更新由你完成。

本文件可以独立运行，不依赖作业 01 的内核变量。


In [ ]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b


## 训练数据：每一行是同一个样本的两个输入和一个目标

下面 x1、x2、y 的 shape 都为 `(6,)`，表示 6 个样本。`predict(x1, x2, ...)` 同时使用两个输入张量，返回 6 个预测值。

| 样本 i | x1[i] | x2[i] | y[i] |
|---|---|---|---|
| 0 | -2 | 1 | -6 |
| 1 | -1 | 0 | -1 |
| 2 | 0 | 2 | -5 |
| 3 | 1 | -1 | 6 |
| 4 | 2 | 3 | -4 |
| 5 | 3 | 1 | 4 |

训练时不要修改输入和目标，也不要直接填写猜测的参数答案。


In [ ]:
x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y = torch.tensor([-6., -1., -5., 6., -4., 4.])
print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y, 'shape:', tuple(y.shape))


## 你的训练区：修改上周模板

创建三个浮点标量参数 w1、w2、b，都启用梯度，并一起交给优化器。每轮调用给定的 predict 函数，同时传入 x1、x2，得到 shape 为 `(6,)` 的 y_pred。

使用绝对值损失：先得到 `(6,)` 的逐样本误差，再用 `.mean()` 得到 shape 为 `()` 的平均损失。只对这个平均损失 backward 一次，然后更新全部参数。

可以从三个参数都为 0、2000 轮、学习率 `0.1 / (epoch ** 0.5)` 开始。这里预测公式是线性的，零初始化与上周的平方参数例子不同。

补完后删除 NotImplementedError。


In [1]:
import torch

# 1. 定义预测函数
def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b

# 2. 训练数据
x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y = torch.tensor([-6., -1., -5., 6., -4., 4.])

# 3. 创建参数（需要梯度）
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

# 4. 创建优化器（SGD，学习率稍大一点以便观察）
# 题目建议初始学习率为 0.1 / (epoch ** 0.5)，这里用 optimizer 管理参数
params = [w1, w2, b]
optimizer = torch.optim.SGD(params, lr=0.1) 

# 5. 训练循环
for epoch in range(1, 2001):
    # 设置本轮学习率（按题目公式衰减）
    current_lr = 0.1 / (epoch ** 0.5)
    for param_group in optimizer.param_groups:
        param_group['lr'] = current_lr

    # 清空旧梯度
    optimizer.zero_grad()

    # 调用 predict，同时使用 x1 和 x2
    y_pred = predict(x1, x2, w1, w2, b)

    # 计算绝对值损失：先得到 (6,) 的逐样本误差，再用 .mean() 得到平均损失
    loss = torch.mean(torch.abs(y_pred - y))

    # 反向传播，更新参数
    loss.backward()
    optimizer.step()

    # 在 torch.no_grad() 中，预测后的参数重新计算预测与损失（用于打印，避免梯度累积）
    if epoch % 200 == 0 or epoch == 1:
        with torch.no_grad():
            # 打印三个参数，x1, x2, y, y_pred, 逐样本损失，平均损失及其 shape
            y_pred_current = predict(x1, x2, w1, w2, b)
            sample_loss = y_pred_current - y
            avg_loss = torch.mean(torch.abs(sample_loss))
            
            print(f"Epoch {epoch}: w1={w1.item():.4f}, w2={w2.item():.4f}, b={b.item():.4f}")
            print(f"  y_pred: {y_pred_current}, y: {y}")
            print(f"  逐样本误差: {sample_loss}, 平均损失: {avg_loss.item():.4f} (shape: {avg_loss.shape})")

C:\Users\OS11\miniforge3\envs\torch-class\Lib\site-packages\torch\_subclasses\functional_tensor.py:276: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


Epoch 1: w1=0.0833, w2=-0.1000, b=-0.0333
  y_pred: tensor([-0.3000, -0.1167, -0.2333,  0.1500, -0.1667,  0.1167]), y: tensor([-6., -1., -5.,  6., -4.,  4.])
  逐样本误差: tensor([ 5.7000,  0.8833,  4.7667, -5.8500,  3.8333, -3.8833]), 平均损失: 4.1528 (shape: torch.Size([]))
Epoch 200: w1=1.7103, w2=-2.4179, b=-0.1738
  y_pred: tensor([-6.0122, -1.8840, -5.0095,  3.9543, -4.0069,  2.5391]), y: tensor([-6., -1., -5.,  6., -4.,  4.])
  逐样本误差: tensor([-0.0122, -0.8840, -0.0095, -2.0457, -0.0069, -1.4609]), 平均损失: 0.7365 (shape: torch.Size([]))
Epoch 400: w1=1.8472, w2=-2.6921, b=0.3865
  y_pred: tensor([-6.0001, -1.4608, -4.9977,  4.9258, -3.9954,  3.2360]), y: tensor([-6., -1., -5.,  6., -4.,  4.])
  逐样本误差: tensor([-6.4850e-05, -4.6076e-01,  2.2788e-03, -1.0742e+00,  4.6227e-03,
        -7.6398e-01]), 平均损失: 0.3843 (shape: torch.Size([]))
Epoch 600: w1=1.9551, w2=-2.9048, b=0.8147
  y_pred: tensor([-6.0002, -1.1403, -4.9949,  5.6746, -3.9895,  3.7751]), y: tensor([-6., -1., -5.,  6., -4.,  4.])
  

## 我的观察与实验

1. 现在一次预测需要几个输入张量？x1 和 x2 与样本数 6 分别是什么关系？（10分）
2. 对照上周：预测公式、参数列表和损失计算分别改了哪里？优化器的基本步骤变了吗？（10分）
3. 记录最终参数、预测和平均损失，说明如何判断训练有效。（20分）
4. 为什么这次 w1=w2=b=0 仍然可以开始学习？（10分）

1.需要两个输入张量（x_1, x_2）;它们都是长度为 6 的向量，每个位置对应一个样本的两个特征。例如第 i 个样本的输入是 (x_1[i], x_2[i])。
2. 预测公式从 w cdot x + b 变成了 w_1 cdot x_1 + w_2 cdot x_2 + b;参数列表从 [w, b] 变成了 [w_1, w_2, b];损失计算逻辑没变（依然是均值），只是预测值来源变了;优化器步骤（清零、前向、算损、反向、步进）完全没变。
3. 训练 2000 轮后，参数会收敛到 w_1 approx 1, w_2 approx -2, b approx 0（根据数据规律：y = 1 cdot x_1 - 2 cdot x_2）;判断有效：平均损失逐渐下降并趋近于 0，预测值 y_pred 与真实值 y 越来越接近。
4. 因为损失函数对参数的梯度在初始时不为零（例如 w_1 的梯度与 x_1 和误差有关），只要数据存在线性关系，梯度下降就能指引参数向正确方向移动。
